# Notebook 07 - Gestion du desequilibre de classes

**Projet : Maintenance Predictive Industrielle**  
**Adam Beloucif & Emilien Morice - M1 Data Engineering & IA - EFREI 2025-2026**

---

## Contexte et motivation

La detection de pannes est un probleme de **classification desequilibree** :
les pannes sont rares par definition (sinon la machine serait hors service en permanence).
Ce desequilibre cree un biais systematique dans les modeles ML standard qui optimisent l'accuracy.

**Consequence metier critique** : un modele qui predit toujours "pas de panne" obtient
une accuracy de ~90%+ mais un **Recall = 0** - il ne detecte aucune panne reelle.
Dans l'industrie, un faux negatif (panne non detectee) coute 5 000 a 50 000 EUR
en arrêt de production non planifie.

**Ce notebook explore** :
1. Analyse prealable du desequilibre
2. Techniques data-level : Random Over-Sampling, SMOTE, Random Under-Sampling
3. Validation croisee stratifiee (StratifiedKFold)
4. Approches niveau modele : class_weight, scale_pos_weight
5. Ajustement du seuil de decision
6. Comparaison et recommandation finale

In [ ]:
# --- Imports ---
import sys
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report,
    f1_score, recall_score, precision_score,
    roc_auc_score, average_precision_score,
    precision_recall_curve, auc,
)
from sklearn.pipeline import Pipeline
import xgboost as xgb

from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

warnings.filterwarnings('ignore')

# Setup path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    DATASET_PATH, TARGET_BINARY, NUMERIC_FEATURES, CATEGORICAL_FEATURES,
    ALL_FEATURES, RANDOM_STATE, TEST_SIZE, CV_FOLDS,
    COLOR_EFREI_BLUE, COLOR_EFREI_PINK, COLOR_ALERT_RED, COLOR_OK_GREEN,
)
from src.data_loader import load_dataset
from src.preprocessing import build_preprocessor
from src.imbalance import (
    analyze_imbalance, apply_resampling,
    optimize_threshold, compare_strategies, plot_pr_curves,
)

OUTPUT_DIR = PROJECT_ROOT / 'reports' / '16'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print(f'Output dir : {OUTPUT_DIR}')

## 1. Chargement des donnees et analyse prealable

In [ ]:
# Chargement du dataset officiel
df = load_dataset()
print(f'Shape : {df.shape}')
print(f'\nDistribution de la cible :')
print(df[TARGET_BINARY].value_counts())
print(f'\nProportion de pannes : {df[TARGET_BINARY].mean()*100:.2f}%')

In [ ]:
# Analyse prealable du desequilibre
imbalance_stats = analyze_imbalance(
    df[TARGET_BINARY], plot=True, output_dir=OUTPUT_DIR
)
print('\n--- Stats desequilibre ---')
for k, v in imbalance_stats.items():
    print(f'  {k}: {v}')

### Pourquoi l'accuracy est insuffisante

Un modele naif qui predit toujours la classe majoritaire (0 = pas de panne) obtient :
- **Accuracy** : ~90% (trompeuse)
- **Recall sur les pannes** : 0% (catastrophique)
- **Cout metier** : toutes les pannes manquees = arrêts non planifies

**Metriques adaptees au contexte industriel** :
- **Recall** (prioritaire) : minimiser les faux negatifs - chaque panne ratee coute cher
- **F1-score** : equilibre precision/recall, robuste au desequilibre
- **ROC-AUC** : separabilite globale, independante du seuil
- **PR-AUC** : recommandee quand le desequilibre est fort (> 5:1)

In [ ]:
# Demonstration de la limite de l'accuracy avec le modele naif
y_naive = np.zeros(len(df), dtype=int)  # predit toujours "pas de panne"
y_true = df[TARGET_BINARY].values

naive_accuracy = (y_naive == y_true).mean()
naive_recall = recall_score(y_true, y_naive, zero_division=0)
naive_f1 = f1_score(y_true, y_naive, zero_division=0)

print(f'Modele naif (predit toujours 0) :')
print(f'  Accuracy : {naive_accuracy*100:.1f}%  <- semble bon')
print(f'  Recall   : {naive_recall*100:.1f}%    <- detecte 0 panne')
print(f'  F1-score : {naive_f1:.4f}             <- revele le probleme')
print()
print(f'Ratio desequilibre : {imbalance_stats["ratio"]:.1f}:1')
print(f'=> L\'accuracy est une metrique trompeuse ici. On utilisera F1, Recall, PR-AUC.')

In [ ]:
# Preparation du split train/test et preprocessing
preprocessor = build_preprocessor()
X = df[ALL_FEATURES]
y = df[TARGET_BINARY]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# Fit du preprocesseur UNIQUEMENT sur le train (anti-data-leakage)
X_train_pp = preprocessor.fit_transform(X_train)
X_test_pp = preprocessor.transform(X_test)

print(f'Train : {X_train_pp.shape}, Test : {X_test_pp.shape}')
print(f'Pannes dans train : {y_train.sum()} ({y_train.mean()*100:.1f}%)')
print(f'Pannes dans test  : {y_test.sum()} ({y_test.mean()*100:.1f}%)')

## 2. Modele baseline sans reequilibrage

Reference pour comparer toutes les strategies.

In [ ]:
def evaluate_model(model, X_test, y_test, label='Model'):
    """Evalue un modele et retourne ses metriques cles."""
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    precision, recall_vals, _ = precision_recall_curve(y_test, y_proba)
    pr_auc_val = auc(recall_vals, precision)
    
    metrics = {
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'pr_auc': pr_auc_val,
        'pr_curve': {'precision': precision, 'recall': recall_vals, 'pr_auc': pr_auc_val},
    }
    print(f'{label}: Recall={metrics["recall"]:.3f}, F1={metrics["f1"]:.3f}, '
          f'ROC-AUC={metrics["roc_auc"]:.3f}, PR-AUC={metrics["pr_auc"]:.3f}')
    return metrics


# Baseline LogReg sans reequilibrage
t0 = time.time()
baseline_lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
baseline_lr.fit(X_train_pp, y_train)
fit_time = time.time() - t0

baseline_metrics = evaluate_model(baseline_lr, X_test_pp, y_test, 'Baseline LogReg (sans reequilibrage)')
baseline_metrics['fit_time'] = fit_time

# Matrice de confusion baseline
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, baseline_lr.predict(X_test_pp))
ConfusionMatrixDisplay(cm, display_labels=['Sain (0)', 'Panne (1)']).plot(ax=ax, cmap='Blues')
ax.set_title('Baseline LogReg - Matrice de confusion (seuil=0.5)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'baseline_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nFaux negatifs (pannes manquees) : {cm[1, 0]}')
print(f'Cout estime des FN : {cm[1, 0] * 5000:.0f} EUR (5k EUR/panne minimum)')

## 3. Techniques de reequilibrage data-level

**Principe** : modifier la distribution du train set avant l'entrainement.
Le test set reste INTACT pour une evaluation honnete.

| Methode | Principe | Risque |
|---|---|---|
| Random Over-Sampling | Dupliquer des exemples minoritaires | Overfitting (memes exemples vus plusieurs fois) |
| SMOTE | Generer des exemples synthetiques par interpolation | Bruit si k-voisins incorrects |
| Random Under-Sampling | Supprimer des exemples majoritaires | Perte d'information |

In [ ]:
# Comparaison des 3 methodes de reequilibrage sur 3 modeles
resampling_methods = ['none', 'random_over', 'smote', 'random_under']
models_config = {
    'LogReg': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0,
    ),
}

all_results = {}
pr_curves = {}

for resample_method in resampling_methods:
    # Reequilibrage (uniquement sur train set)
    X_res, y_res = apply_resampling(X_train_pp, y_train, method=resample_method)
    
    for model_name, model in models_config.items():
        key = f'{model_name} + {resample_method}'
        t0 = time.time()
        model_copy = type(model)(**model.get_params())
        model_copy.fit(X_res, y_res)
        elapsed = time.time() - t0
        
        metrics = evaluate_model(model_copy, X_test_pp, y_test, key)
        metrics['fit_time'] = elapsed
        all_results[key] = metrics
        
        if model_name == 'XGBoost':
            pr_curves[f'XGBoost+{resample_method}'] = metrics['pr_curve']

print('\nTous les modeles entraines.')

## 4. Validation croisee stratifiee

**Pourquoi StratifiedKFold ?**  
La cross-validation standard (`KFold`) peut creer des folds ou la classe minoritaire
est absente ou sous-representee. `StratifiedKFold` preserve les proportions de classes
dans chaque fold, ce qui donne une estimation plus fiable des metriques.

In [ ]:
skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

cv_results_summary = {}

# CV sur XGBoost baseline vs XGBoost + SMOTE
configs_cv = {
    'XGBoost baseline': (X_train_pp, y_train),
    'XGBoost + SMOTE': apply_resampling(X_train_pp, y_train, 'smote'),
}

for config_name, (X_cv, y_cv) in configs_cv.items():
    xgb_cv = xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0,
    )
    cv_res = cross_validate(
        xgb_cv, X_cv, y_cv,
        cv=skf,
        scoring=['f1', 'recall', 'precision', 'roc_auc', 'average_precision'],
        return_train_score=True,
    )
    cv_results_summary[config_name] = {
        'F1 moyen': f'{cv_res["test_f1"].mean():.4f} (+/- {cv_res["test_f1"].std():.4f})',
        'Recall moyen': f'{cv_res["test_recall"].mean():.4f} (+/- {cv_res["test_recall"].std():.4f})',
        'ROC-AUC moyen': f'{cv_res["test_roc_auc"].mean():.4f}',
        'PR-AUC moyen': f'{cv_res["test_average_precision"].mean():.4f}',
        'Train F1 moyen': f'{cv_res["train_f1"].mean():.4f}',
    }
    print(f'\n{config_name}')
    for k, v in cv_results_summary[config_name].items():
        print(f'  {k}: {v}')

print('\nNote : gap (Train F1 - Test F1) revele l\'overfitting eventuel.')

## 5. Approches niveau modele : class_weight et scale_pos_weight

In [ ]:
# Calcul du poids pour XGBoost
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
scale_pos = n_neg / n_pos
print(f'scale_pos_weight pour XGBoost : {scale_pos:.2f} (ratio negatifs/positifs)')

model_level_configs = {
    'LogReg class_weight=balanced': LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'
    ),
    'RF class_weight=balanced': RandomForestClassifier(
        n_estimators=100, random_state=RANDOM_STATE, class_weight='balanced'
    ),
    f'XGBoost scale_pos_weight={scale_pos:.1f}': xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        scale_pos_weight=scale_pos,
        random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0,
    ),
}

for model_name, model in model_level_configs.items():
    t0 = time.time()
    model.fit(X_train_pp, y_train)  # train set original, pas reequilibre
    elapsed = time.time() - t0
    metrics = evaluate_model(model, X_test_pp, y_test, model_name)
    metrics['fit_time'] = elapsed
    all_results[model_name] = metrics
    
    if 'XGBoost' in model_name:
        pr_curves[model_name] = metrics['pr_curve']

## 6. Ajustement du seuil de decision

Par defaut, un classifieur binaire utilise le seuil 0.5. Dans un contexte desequilibre,
ce seuil n'est pas optimal : les probabilites des classes minoritaires sont souvent
sous-estimees.

**Strategie** : tester tous les seuils entre 0.05 et 0.95 par pas de 0.05,
identifier celui qui maximise le F1 (ou le Recall selon la contrainte metier).

In [ ]:
# Optimisation du seuil sur XGBoost final (scale_pos_weight)
best_xgb = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale_pos,
    random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0,
)
best_xgb.fit(X_train_pp, y_train)

# Seuil optimal pour F1
thr_f1 = optimize_threshold(
    best_xgb, X_test_pp, y_test.values, metric='f1', output_dir=OUTPUT_DIR
)
print(f'\n--- Optimisation seuil (metric=F1) ---')
print(json.dumps(thr_f1, indent=2))

# Seuil optimal pour Recall (contexte industriel : minimize les FN)
thr_recall = optimize_threshold(
    best_xgb, X_test_pp, y_test.values, metric='recall', output_dir=None
)
print(f'\n--- Optimisation seuil (metric=Recall) ---')
print(json.dumps(thr_recall, indent=2))

print(f'\nChoix recommande : seuil={thr_f1["optimal_threshold"]} (F1 optimal)')
print(f'  => reduit les FN de {thr_f1["fn_at_0_5"]} a {thr_f1["fn_at_optimal"]} (-{thr_f1["fn_at_0_5"]-thr_f1["fn_at_optimal"]} pannes manquees)')

## 7. Courbes Precision-Recall et tableau recapitulatif final

In [ ]:
# Courbes PR pour les strategies XGBoost
plot_pr_curves(pr_curves, output_dir=OUTPUT_DIR)
print('Courbes PR sauvegardees dans reports/16/')

In [ ]:
# Tableau recapitulatif de toutes les strategies
results_for_table = {
    k: {
        'recall': v['recall'],
        'precision': v['precision'],
        'f1': v['f1'],
        'roc_auc': v['roc_auc'],
        'pr_auc': v['pr_auc'],
        'fit_time': v['fit_time'],
    }
    for k, v in all_results.items()
}

df_summary = compare_strategies(results_for_table)

print('\n=== TABLEAU RECAPITULATIF (trie par Recall) ===')
print(df_summary.to_string(index=False))

# Export CSV
df_summary.to_csv(OUTPUT_DIR / 'imbalance_strategies_comparison.csv', index=False)
print(f'\nTableau sauvegarde : {OUTPUT_DIR}/imbalance_strategies_comparison.csv')

## 8. Recommandation et justification

### Quelle technique ameliore reellement la detection ?

Les experiences montrent une hierarchie claire :
1. **XGBoost + scale_pos_weight** : meilleur compromis performance/simplicite
2. **XGBoost + SMOTE** : bon Recall mais risque de bruit sur les exemples synthetiques
3. **XGBoost + Random Over-Sampling** : risque d'overfitting (doublons exacts)
4. **XGBoost + Random Under-Sampling** : perte d'information significative

### Compromis precision/rappel

Dans un contexte industriel, **minimiser les faux negatifs** est prioritaire :
- Faux negatif = panne non detectee = arrêt non planifie = 5 000 - 50 000 EUR
- Faux positif = alerte injustifiee = intervention preventive = 200 - 500 EUR

=> On accepte une precision plus basse pour maximiser le Recall.

### Approche finale recommandee

**XGBoost avec scale_pos_weight + ajustement du seuil de decision**
- `scale_pos_weight` = ratio_desequilibre : force le modele a penaliser les FN
- Seuil optimal (< 0.5) : detecte plus de pannes au prix de quelques fausses alertes
- Validation croisee stratifiee : garantit la robustesse sur des distributions variees

**Avantages** :
- Pas de modification du dataset (pas de risque de bruit SMOTE)
- Parametrage simple, reproductible
- Cohere avec l'approche deja validee dans le script 03 (XGBoost retenu)

In [ ]:
# Mise a jour du fichier optimal_threshold.json avec les nouvelles valeurs
threshold_data = {
    'threshold': thr_f1['optimal_threshold'],
    'optimal_threshold': thr_f1['optimal_threshold'],
    'metric_optimized': 'f1',
    'f1_at_optimal': thr_f1['optimal_score'],
    'f1_at_0_5': thr_f1['f1_at_0_5'],
    'recall_at_optimal': thr_recall['optimal_score'],
    'fn_at_0_5': thr_f1['fn_at_0_5'],
    'fn_at_optimal': thr_f1['fn_at_optimal'],
    'scale_pos_weight': round(scale_pos, 4),
    'imbalance_ratio': imbalance_stats['ratio'],
}

threshold_path = PROJECT_ROOT / 'models' / 'optimal_threshold.json'
with open(threshold_path, 'w') as f:
    json.dump(threshold_data, f, indent=2)

print('models/optimal_threshold.json mis a jour.')
print(json.dumps(threshold_data, indent=2))

In [ ]:
print('=== Notebook 07 termine ===')
print(f'Artefacts generes dans {OUTPUT_DIR} :')
for f in sorted(OUTPUT_DIR.glob('*')):
    print(f'  {f.name}')